# Experiment 00 — P3 Baseline: CE-Only Fine-Tuning (No Distillation)

## Setup

| Item | Value |
|------|-------|
| **Pair** | P3 — `yolov8x-cls` (teacher) → `yolov8m-cls` (student) |
| **Dataset** | CIFAR10 (50 000 train / 10 000 val, 32×32, no resize) |
| **Pruning** | 50% unstructured magnitude (L1-norm, local scope) |
| **Training** | CE-only (α=0.0, no teacher signal) |
| **Epochs** | 5 |
| **Optimizer** | Adam, lr=1e-4 |
| **Batch size** | 128 |
| **Seed** | 42 |

### Purpose

This notebook establishes the **CE-only fine-tuning baseline** for pair P3.  
The pruned `yolov8m-cls` student is trained using only hard-label cross-entropy loss  
(no teacher supervision). All KD experiment notebooks (`exp_01`–`exp_25`) compare  
against this baseline to measure the incremental benefit of knowledge distillation.

### Conditions evaluated

1. **Teacher** — CIFAR10 fine-tuned `yolov8x-cls`, untouched  
2. **Pruned (no train)** — Student immediately after 50% pruning, no further training  
3. **Fine-tuned (CE only)** — Pruned student trained 5 epochs with CE loss

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
EXPERIMENT_ID = "exp_00"
PAIR = "P3"

TEACHER_CHECKPOINT = "data/cifar10_yolov8x_cls/runs/yolov8x_cls_cifar10_finetune/weights/best.pt"
STUDENT_CHECKPOINT = "data/cifar10_yolov8m_cls/runs/yolov8m_cls_cifar10_finetune/weights/best.pt"
TEACHER_CFG = "yolov8x-cls.yaml"
STUDENT_CFG = "yolov8m-cls.yaml"

DEVICE = "cuda"
DATA_ROOT = "./data"

IMAGE_SIZE = 32
BATCH_SIZE = 128

PRUNE_SPARSITY = 0.50
EPOCHS = 5
LR = 1e-4

SEED = 42

RESULTS_CSV = "YOLO_logitKD_experiments/results.csv"

## Imports and setup

In [ ]:
import copy
import csv
import random
import sys
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo.yolov8 import MaseYoloClassificationModel, patch_yolo

from mase_kd.core.losses import hard_label_ce_loss
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

if DEVICE == "cuda" and not torch.cuda.is_available():
    DEVICE = "cpu"

torch.manual_seed(SEED)
random.seed(SEED)

print(f"Repo root: {repo_root}")
print(f"Device:    {DEVICE}")

## CIFAR10 dataloaders

In [ ]:
cifar_transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True,  transform=cifar_transform, download=True)
val_dataset   = datasets.CIFAR10(root=DATA_ROOT, train=False, transform=cifar_transform, download=True)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True, drop_last=True,
)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

## Load CIFAR10-fine-tuned teacher (yolov8x-cls)

In [ ]:
ultra_teacher = YOLO(TEACHER_CHECKPOINT)
nc = ultra_teacher.model.yaml.get("nc", 10)

teacher_cls_model = MaseYoloClassificationModel(cfg=TEACHER_CFG, nc=nc)
teacher_cls_model = patch_yolo(teacher_cls_model)
teacher_cls_model.load_state_dict(ultra_teacher.model.state_dict())
teacher_cls_model = teacher_cls_model.to(DEVICE)
teacher_cls_model.eval()

print(f"Teacher loaded: {TEACHER_CHECKPOINT}  (nc={nc})")

## Build student from checkpoint and prune (50% sparsity)

In [ ]:
ultra_student = YOLO(STUDENT_CHECKPOINT)

student_seed = MaseYoloClassificationModel(cfg=STUDENT_CFG, nc=nc)
student_seed = patch_yolo(student_seed)
student_seed.load_state_dict(ultra_student.model.state_dict())

mg = MaseGraph(student_seed)
mg, _ = passes.init_metadata_analysis_pass(mg)

trace_input = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
placeholder_names = [n.name for n in mg.fx_graph.nodes if n.op == "placeholder"]
dummy_in = {name: trace_input for name in placeholder_names}

mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_in})

pruning_config = {
    "weight":     {"sparsity": PRUNE_SPARSITY, "method": "l1-norm", "scope": "local"},
    "activation": {"sparsity": PRUNE_SPARSITY, "method": "l1-norm", "scope": "local"},
}
mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)

student_cls_model = mg.model.to(DEVICE)
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(DEVICE)
pruned_no_kd_model.eval()

print(f"Student loaded: {STUDENT_CHECKPOINT}")
print(f"Pruning complete ({PRUNE_SPARSITY*100:.0f}% sparsity)")

## Evaluate pruned model (no training)

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    batches = samples = correct = 0
    total_ce = total_ms = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue
        total_ms += (t1 - t0) * 1000.0
        batches += 1
        samples += images.shape[0]
        if logits.shape[1] > int(labels.max().item()):
            correct += int((logits.argmax(dim=1) == labels).sum().item())
            total_ce += hard_label_ce_loss(logits, labels).item()
    return {
        "top1_acc": correct / max(samples, 1),
        "avg_ce_loss": total_ce / max(batches, 1),
        "avg_forward_ms": total_ms / max(batches, 1),
        "samples": samples,
        "batches": batches,
    }

pruned_metrics = evaluate_model(pruned_no_kd_model, val_loader, DEVICE)
print(f"Pruned (no train) — top1: {pruned_metrics['top1_acc']*100:.2f}%  CE: {pruned_metrics['avg_ce_loss']:.4f}")

## Fine-tune pruned student with CE loss only (no distillation)

In [ ]:
ce_model = copy.deepcopy(pruned_no_kd_model).to(DEVICE)
ce_optimizer = torch.optim.Adam(ce_model.parameters(), lr=LR)

ce_loss_history = []
ce_top1_history = []

for epoch in range(1, EPOCHS + 1):
    ce_model.train()
    epoch_loss = 0.0
    num_batches = len(train_loader)
    for batch_idx, (images, labels) in enumerate(train_loader, start=1):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        ce_optimizer.zero_grad(set_to_none=True)
        outputs = ce_model(images)
        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        loss = hard_label_ce_loss(logits, labels)
        loss.backward()
        ce_optimizer.step()
        epoch_loss += loss.item()

        with torch.no_grad():
            top1 = float((logits.argmax(dim=1) == labels).float().mean())
        ce_loss_history.append(loss.item())
        ce_top1_history.append(top1)

        if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == num_batches:
            print(f"  Epoch {epoch}/{EPOCHS} | Batch {batch_idx:04d}/{num_batches} | "
                  f"loss={loss.item():.6f} | top1={top1*100:.2f}%")
    print(f"Epoch {epoch} avg loss: {epoch_loss / num_batches:.6f}")

ce_metrics = evaluate_model(ce_model, val_loader, DEVICE)
print(f"\nCE-only fine-tuned — top1: {ce_metrics['top1_acc']*100:.2f}%  CE: {ce_metrics['avg_ce_loss']:.4f}")

## Evaluate teacher

In [ ]:
teacher_metrics = evaluate_model(teacher_cls_model, val_loader, DEVICE)
print(f"Teacher — top1: {teacher_metrics['top1_acc']*100:.2f}%  CE: {teacher_metrics['avg_ce_loss']:.4f}")

## Summary

In [ ]:
print(f"{'Model':<40} {'Top-1':>8} {'CE Loss':>10}")
print(f"{'─'*40} {'─'*8} {'─'*10}")
print(f"{'Teacher (yolov8x-cls)':<40} {teacher_metrics['top1_acc']*100:>7.2f}% {teacher_metrics['avg_ce_loss']:>10.4f}")
print(f"{'Pruned (no train, 50%)':<40} {pruned_metrics['top1_acc']*100:>7.2f}% {pruned_metrics['avg_ce_loss']:>10.4f}")
print(f"{'CE-only (5 epochs)':<40} {ce_metrics['top1_acc']*100:>7.2f}% {ce_metrics['avg_ce_loss']:>10.4f}")

## Save results to CSV

In [ ]:
row = {
    "pair": PAIR,
    "alpha": 0.0,
    "temperature": "",
    "lr": LR,
    "epochs": EPOCHS,
    "sparsity": PRUNE_SPARSITY,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "teacher_top1": round(teacher_metrics["top1_acc"] * 100, 4),
    "pruned_top1": round(pruned_metrics["top1_acc"] * 100, 4),
    "ce_only_top1": round(ce_metrics["top1_acc"] * 100, 4),
    "kd_top1": "",
    "kd_gain_vs_ce": "",
    "teacher_ce_loss": round(teacher_metrics["avg_ce_loss"], 6),
    "pruned_ce_loss": round(pruned_metrics["avg_ce_loss"], 6),
    "ce_only_ce_loss": round(ce_metrics["avg_ce_loss"], 6),
    "kd_ce_loss": "",
    "val_kd_loss": "",
    "notebook": EXPERIMENT_ID,
}

csv_path = Path(RESULTS_CSV)
file_exists = csv_path.exists() and csv_path.stat().st_size > 0
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Results appended to {RESULTS_CSV}")

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{EXPERIMENT_ID} — CE-Only Fine-Tuning (P3, 50% pruned)", fontsize=13)

batches_per_epoch = len(ce_loss_history) // EPOCHS

ax = axes[0]
ax.plot(ce_loss_history, alpha=0.8, linewidth=0.8)
for e in range(1, EPOCHS):
    ax.axvline(e * batches_per_epoch, color="gray", ls="--", lw=0.6, alpha=0.5)
ax.set_xlabel("Batch")
ax.set_ylabel("CE Loss")
ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(ce_top1_history, alpha=0.8, linewidth=0.8)
for e in range(1, EPOCHS):
    ax.axvline(e * batches_per_epoch, color="gray", ls="--", lw=0.6, alpha=0.5)
ax.set_xlabel("Batch")
ax.set_ylabel("Top-1 Accuracy")
ax.set_title("Training Top-1")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%"))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()